# WikiGraph Agent Demo
## LLM Wiki Philosophy + LightRAG Engine

LLM Wiki의 "지식이 자라는" 철학을 LightRAG의 그래프 엔진 위에서 실현합니다.

1. **INGEST** — 문서 삽입 + 검증
2. **QUERY** — 그래프 즉시 검색 + 쿼리 로그
3. **EVOLVE** — 쿼리 패턴 분석 → 지식 자동 진화
4. **LINT** — 그래프 건강검진

## 1. Setup

In [1]:
import sys, os, shutil
import numpy as np
sys.path.insert(0, '..')
sys.path.insert(0, '.')

from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

LLM_BASE_URL = 'http://222.117.133.162:30010/v1'
LLM_MODEL    = 'qwen-task-pool'
LLM_API_KEY  = 'asdf'
EMBED_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM    = 384
WORK_DIR     = '/tmp/wikigraph_demo'

print(f'Loading embedding model: {EMBED_MODEL} ...')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Done.')

async def llm_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        LLM_MODEL, prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        **kwargs,
    )

async def embed_func(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(f'LLM: {LLM_BASE_URL} ({LLM_MODEL})')
print(f'Embedding: {EMBED_MODEL} (local)')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Done.
LLM: http://222.117.133.162:30010/v1 (qwen-task-pool)
Embedding: sentence-transformers/all-MiniLM-L6-v2 (local)


## 2. LightRAG + WikiGraph Agent 초기화

In [ ]:
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

rag = LightRAG(
    working_dir=WORK_DIR,
    llm_model_func=llm_func,
    llm_model_max_async=1,
    default_llm_timeout=300,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=8192,
        func=embed_func,
    ),
    addon_params={
        'enable_hybrid_search': True,
        'hybrid_search_mode': 'hybrid',
    },
)
await rag.initialize_storages()

from wikigraph.agent import WikiGraphAgent
from wikigraph.config import WikiGraphConfig

config = WikiGraphConfig(working_dir=WORK_DIR, auto_evolve_interval=5)
agent = WikiGraphAgent(rag, config, llm_func=llm_func)

print(f'LightRAG ready: {WORK_DIR}')
print(f'WikiGraph Agent initialized')
print(f'Hybrid Search: {rag._addon_params["enable_hybrid_search"]}')
print(f'Auto-evolve interval: every {config.auto_evolve_interval} queries')

## 3. INGEST — 문서 삽입

LLM Wiki처럼 문서를 넣으면 자동으로:
- 청킹 → 임베딩 → 엔티티/관계 추출 → 그래프 저장 → BM25 인덱싱

LLM Wiki와 다른 점: **증분 업데이트** — 기존 그래프를 재구축하지 않음

여기서는 LightRAG 프로젝트에 대한 요약 텍스트를 직접 삽입합니다.

In [ ]:
# 짧은 텍스트로 빠르게 테스트 (LLM 서버 타임아웃 방지)
doc1 = """LightRAG is a graph-based Retrieval-Augmented Generation framework developed by HKUDS.
It extracts entities and relationships from documents to build a knowledge graph.
The system supports multiple query modes: local, global, hybrid, naive, and mix.
LightRAG uses vector embeddings stored in NanoVectorDB for similarity search.
The frontend WebUI is built with React, TypeScript, and Bun as the build tool.
Vite is used as the development server with hot module replacement."""

doc2 = """BM25 is a keyword-based ranking function used in information retrieval.
It uses term frequency and inverse document frequency to score documents.
Reciprocal Rank Fusion (RRF) combines results from multiple search methods.
The formula is RRF(d) = sum(1/(k + rank)), where k is typically 60.
Hybrid search combines vector similarity search with BM25 keyword matching.
This approach improves recall for proper nouns and short keyword queries."""

print(f'Document 1: {len(doc1)} chars (LightRAG overview)')
print(f'Document 2: {len(doc2)} chars (BM25/Hybrid search)')

print(f'\nIngesting 2 documents...')
result = await agent.ingest([doc1, doc2], file_paths=['lightrag_overview.txt', 'bm25_hybrid.txt'])
for msg in result['messages']:
    print(f'  {msg}')

## 4. QUERY — 쿼리 + 자동 평가

쿼리 결과를 자동 평가하고, 품질이 낮으면 EVOLVE를 트리거합니다.
쿼리 로그가 축적되어 나중에 EVOLVE의 입력이 됩니다.

In [ ]:
queries = [
    'What is LightRAG?',
    'How does BM25 work?',
    'What is hybrid search?',
    'What query modes exist?',
    'What is RRF?',
]

for q in queries:
    print(f'\n{"="*60}')
    result = await agent.query(q)
    for msg in result['messages']:
        print(f'  {msg}')
    if result.get('evolved'):
        print('  >>> Knowledge evolved!')

print(f'\n--- Agent Stats ---')
stats = agent.stats
print(f'Total queries: {stats["total_queries"]}')
print(f'Tracked entities: {stats["tracked_entities"]}')

## 5. EVOLVE — 지식 진화

쿼리 로그를 분석하여:
1. 자주 함께 검색되는 엔티티 → 새 관계 생성
2. 실패한 쿼리 → 지식 갭 채우기
3. 멀티홉 경로 → 단축 관계 생성

In [5]:
print('Running EVOLVE...')
result = await agent.evolve()
for msg in result['messages'][-10:]:
    print(f'  {msg}')
print(f'\nMutations applied: {result["applied"]}')

Running EVOLVE...

Mutations applied: 0


## 6. LINT — 그래프 건강검진

LLM Wiki는 전체 wiki를 LLM으로 스캔해야 하지만,
WikiGraph는 **그래프 알고리즘**으로 0토큰 탐지합니다.

In [6]:
print('Running LINT...')
result = await agent.lint()
for msg in result['messages'][-10:]:
    print(f'  {msg}')

if result['findings']:
    print(f'\n--- Findings ({len(result["findings"])}) ---')
    for f in result['findings'][:10]:
        print(f'  [{f["severity"]}] {f["finding_type"]}: {f["entity_name"]}')
        print(f'    {f["details"]}')
        print(f'    Action: {f["suggested_action"]}')

Running LINT...
  LINT: scanning 0 entities
  LINT: no issues found


## 7. 그래프 시각화

In [7]:
import networkx as nx

graph_path = os.path.join(WORK_DIR, 'graph_chunk_entity_relation.graphml')
if os.path.exists(graph_path):
    G = nx.read_graphml(graph_path)
    print(f'Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

    evolved_edges = [(u, v) for u, v, d in G.edges(data=True) if 'wikigraph_evolve' in d.get('source_id', '')]
    print(f'Evolved edges: {len(evolved_edges)}')
    for u, v in evolved_edges[:10]:
        print(f'  {u} → {v} (auto-generated by EVOLVE)')

    print(f'\nTop entities by degree:')
    degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    for name, deg in degrees[:10]:
        print(f'  {name:30s} degree={deg}')
else:
    print('Graph file not found — INGEST may have failed')

Graph file not found — INGEST may have failed


## 8. Cleanup

In [8]:
await rag.finalize_storages()
print('Done.')

INFO: [] chunks flush: embedding 2 vectors in 1 batch(es) (batch_num=10)


INFO: Successfully finalized 12 storages


Done.


## Summary

| 차원 | LLM Wiki v2 | WikiGraph Agent |
|---|---|---|
| **스케일** | ~1000페이지 | 수만 노드 |
| **업데이트** | 페이지 15개 재작성 | 노드/엣지 증분 |
| **검색** | 풀컨텍스트 로딩 | 그래프+벡터+BM25 |
| **오류** | 전파됨 | 원본 보존+검증 |
| **건강검진** | LLM 전체 스캔 | 그래프 알고리즘 |
| **지식 진화** | LLM 재작성 | 쿼리 패턴 기반 자동 |

```
WikiGraph = LLM Wiki의 철학 + LightRAG의 엔진
         = 스케일러블하게 자라는 지식 그래프
```